In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from shared_utils import *
import monai
from monai.networks.nets import SwinUNETR

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'MONAI: {monai.__version__}')


shared_utils.py yüklendi. [Q1 | Sensitivity-First | TTA | ClinicalFocalLoss]
Device: cuda
MONAI: 1.6.0


In [2]:
DATA_ROOT = Path(r"/home/zera/Downloads/Appendiks varyasyon3 DS-20260713T105239Z-2-001/Appendiks varyasyon3 DS")
BASE_DIR  = DATA_ROOT / "segformer/experiments/swinunetr_linearprobe"
BASE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {**SHARED_CONFIG,
    "model_name":     "swinunetr_lp",
    "feature_size":   48,
    "n_epochs":       80,
    "patience":       20,
    "swa_start":      40,
    "lr":             1e-3,    # Backbone donduruldu → head hızlı öğrenebilir
    "weight_decay":   1e-2,
    "warmup_epochs":  5,
    "min_sens_floor": 0.80,
    "min_spec_floor": 0.50,
    "sensitivity_first": True,
    "focal_gamma":    2.0,
    "label_smoothing": 0.05,
}

test_df = pd.read_csv(DATA_ROOT / "segformer/datas" / "external_test_set.csv")
print(f"External Test: {len(test_df)}")
print(f"LR: {CONFIG['lr']} | Backbone: TAMAMEN DONDURULMUŞ")


External Test: 37
LR: 0.001 | Backbone: TAMAMEN DONDURULMUŞ


In [3]:
# ============================================================
# SwinUNETR Linear Probe
# Strateji: Backbone TAMAMEN dondurulmuş (requires_grad=False)
# Sadece küçük head eğitiliyor: ~50K parametre
# 165 hasta için doğru seçim: overfitting imkansız
# ============================================================
class SwinLinearProbe(nn.Module):
    def __init__(self, num_classes=2, feature_size=48):
        super().__init__()
        self.backbone = SwinUNETR(
            in_channels=1, out_channels=14,
            feature_size=feature_size,
            use_checkpoint=True, spatial_dims=3,
        )
        # Backbone TAMAMEN donduruldu
        for p in self.backbone.parameters():
            p.requires_grad = False

        dim4 = feature_size * 16  # 768 — son SwinViT katmanı (en semantik)
        dim3 = feature_size * 8   # 384 — sondan bir önceki

        self.gap = nn.AdaptiveAvgPool3d(1)

        # Multi-scale: son 2 katman → LayerNorm → Dropout → 64 → 2
        # LayerNorm batch_size=4'te BatchNorm'dan çok daha stabil
        self.proj4 = nn.Linear(dim4, 64)
        self.proj3 = nn.Linear(dim3, 32)
        self.head = nn.Sequential(
            nn.LayerNorm(96),
            nn.Dropout(0.50),
            nn.Linear(96, num_classes),
        )

    def forward(self, x):
        with torch.no_grad():  # Backbone gradyan yok — bellek de korunur
            hidden = self.backbone.swinViT(x, self.backbone.normalize)
        # Son 2 katman
        f3 = self.gap(hidden[3]).flatten(1)  # [B, 384]
        f4 = self.gap(hidden[4]).flatten(1)  # [B, 768]
        p3 = F.gelu(self.proj3(f3))  # [B, 32]
        p4 = F.gelu(self.proj4(f4))  # [B, 64]
        feat = torch.cat([p3, p4], dim=1)  # [B, 96]
        return self.head(feat)


def load_pretrained_swin(model, ckpt_path=None):
    if ckpt_path is None:
        candidates = [
            os.path.expanduser("~/models/model_swinvit.pt"),
            str(DATA_ROOT / "segformer/model_swinvit.pt"),
        ]
        ckpt_path = next((p for p in candidates if Path(p).exists()), None)
    if not ckpt_path or not Path(ckpt_path).exists():
        print("[WARN] Pretrained ağırlık bulunamadı.")
        return model
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if "state_dict" in sd: sd = sd["state_dict"]
    target = model.backbone.swinViT.state_dict()
    new_sd, loaded, skipped = {}, 0, 0
    for k, v in sd.items():
        k2 = k.replace("swinViT.", "").replace("module.", "")
        if k2 in target and target[k2].shape == v.shape:
            new_sd[k2] = v; loaded += 1
        else:
            skipped += 1
    model.backbone.swinViT.load_state_dict(new_sd, strict=False)
    print(f"  Pretrained: {loaded} katman yüklendi, {skipped} atlandı.")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Eğitilebilir: {trainable:,} / {total:,} parametre ({100*trainable/total:.1f}%)")
    return model


In [4]:
from torch.optim.swa_utils import AveragedModel, SWALR

def run_fold(train_df, val_df, fold_idx, config, output_dir):
    def set_seed(s):
        torch.manual_seed(s); np.random.seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    set_seed(config["random_seed"] + fold_idx)

    fold_dir = Path(output_dir) / f"fold_{fold_idx}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_ds = AppendixH5Dataset(train_df, augment=True,  config=config)
    val_ds   = AppendixH5Dataset(val_df,   augment=False, config=config)

    labels = train_df["label"].values.astype(int)
    class_counts = np.bincount(labels)
    print(f"  [Fold {fold_idx}] {class_counts} | LP: backbone dondurulmuş")

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"],
                              shuffle=True, num_workers=config["num_workers"],
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=config["batch_size"],
                              shuffle=False, num_workers=config["num_workers"], pin_memory=True)

    model = SwinLinearProbe(feature_size=config["feature_size"]).to(DEVICE)
    model = load_pretrained_swin(model)

    # Sadece head parametreleri optimizer'a veriliyor
    head_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(head_params, lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = get_warmup_cosine_scheduler(optimizer, config["warmup_epochs"], config["n_epochs"])

    criterion = ClinicalFocalLoss(
        pos_weight=config.get("pos_weight", 1.0),
        gamma=config.get("focal_gamma", 2.0),
        smoothing=config.get("label_smoothing", 0.05)
    )

    swa_model = AveragedModel(model)
    swa_start = config.get("swa_start", 40)
    swa_sched = SWALR(optimizer, swa_lr=config["lr"] * 0.05)

    best_score, patience_cnt = -1.0, 0
    history = []

    for epoch in range(1, config["n_epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, val_auc, val_acc, val_f1, pred_df = evaluate_model(
            model, val_loader, criterion, DEVICE)

        y_true = pred_df["label"].values
        y_prob = pred_df["prob_mucinous"].values
        threshold = float(pred_df["_threshold_used"].iloc[0]) if "_threshold_used" in pred_df.columns else 0.5
        y_pred = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        sens = tp / (tp + fn + 1e-9)
        spec = tn / (tn + fp + 1e-9)
        from sklearn.metrics import f1_score as _f1
        val_f1_thr = float(_f1(y_true, y_pred, zero_division=0))
        composite  = clinical_composite(sens, val_auc, spec, f1=val_f1_thr)

        print(f"    [Prob] min={y_prob.min():.3f} max={y_prob.max():.3f} | TP={tp} FP={fp} FN={fn} TN={tn}")
        print(f"[lp | fold {fold_idx} | epoch {epoch:03d}] "
              f"train={train_loss:.4f} val={val_loss:.4f} auc={val_auc:.4f} thr={threshold:.3f}")
        print(f"  [Metrics] AUC:{val_auc:.3f} F1:{val_f1_thr:.3f} "
              f"SENS:{sens:.3f} SPEC:{spec:.3f} | COMPOSITE:{composite:.4f}")

        history.append({"epoch": epoch, "train_loss": train_loss,
                        "val_loss": val_loss, "auc": val_auc,
                        "sens": sens, "spec": spec, "f1": val_f1_thr,
                        "composite": composite})

        if epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_sched.step()
        else:
            scheduler.step()

        # ── Çift kısıt model seçimi ────────────────────────────────────
        min_sens = config.get("min_sens_floor", 0.80)
        min_spec = config.get("min_spec_floor", 0.50)
        if sens >= min_sens and spec >= min_spec and composite > best_score:
            best_score = composite
            torch.save(model.state_dict(), fold_dir / "best_model.pt")
            patience_cnt = 0
            print(f"    ★ SAVED (SENS={sens:.3f} SPEC={spec:.3f} F1={val_f1_thr:.3f} composite={composite:.4f})")
        else:
            patience_cnt += 1
            if patience_cnt >= config["patience"]:
                print(f"  Early stopping @ {epoch} | best={best_score:.4f}")
                break

    # SWA finalize
    if epoch >= swa_start:
        print("  SWA finalize...")
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)
        _, auc_s, _, _, pdf_s = evaluate_model(swa_model, val_loader, criterion, DEVICE)
        y_t = pdf_s["label"].values; y_p = pdf_s["prob_mucinous"].values
        thr_s = float(pdf_s["_threshold_used"].iloc[0])
        yp_s  = (y_p >= thr_s).astype(int)
        tn_,fp_,fn_,tp_ = confusion_matrix(y_t, yp_s, labels=[0,1]).ravel()
        swa_c = clinical_composite(tp_/(tp_+fn_+1e-9), auc_s, tn_/(tn_+fp_+1e-9),
                                   f1=float(_f1(y_t, yp_s, zero_division=0)))
        print(f"  SWA composite: {swa_c:.4f} | Best: {best_score:.4f}")
        if swa_c > best_score:
            torch.save(swa_model.state_dict(), fold_dir / "best_model.pt")
            print("  SWA seçildi!")

    if best_score < 0:
        print("  [WARN] Kısıt sağlanamadı — son model fallback")
        torch.save(model.state_dict(), fold_dir / "best_model.pt")

    # Final eval
    model.load_state_dict(torch.load(fold_dir / "best_model.pt", map_location=DEVICE, weights_only=False))
    _, auc_f, _, _, pdf_f = evaluate_model(model, val_loader, criterion, DEVICE)
    youden_thr, _ = find_youden_threshold(pdf_f["label"].values, pdf_f["prob_mucinous"].values)
    ci = compute_bootstrap_ci(pdf_f["label"].values, pdf_f["prob_mucinous"].values, youden_thr)
    m_f, cm_f, _ = compute_binary_metrics(pdf_f["label"].values, pdf_f["prob_mucinous"].values, youden_thr)
    print_full_metrics_table(m_f, ci, f"SwinLP Fold {fold_idx}", f"Youden {youden_thr:.3f}")
    plot_confusion_matrix(cm_f, f"Fold {fold_idx}", save_path=fold_dir / f"cm_{fold_idx}.png")
    pdf_f.to_csv(fold_dir / "val_predictions.csv", index=False)

    import json as _j
    with open(fold_dir / "history.json", "w") as fh: _j.dump(history, fh)
    return m_f, ci, pdf_f, history


In [ ]:
import json

all_preds, all_metrics = [], []

for fold_idx in range(1, 6):
    train_df = pd.read_csv(DATA_ROOT / "segformer/datas" / f"fold_{fold_idx}_train.csv")
    val_df   = pd.read_csv(DATA_ROOT / "segformer/datas" / f"fold_{fold_idx}_val.csv")
    print(f"{'='*70}FOLD {fold_idx}/5{'='*70}")
    m, ci, pred, hist = run_fold(train_df, val_df, fold_idx, CONFIG, BASE_DIR)
    all_preds.append(pred)
    all_metrics.append(m)

print("\n5-Fold CV tamamlandı.")
print(f"Ortalama AUC: {np.mean([m['auc_roc'] for m in all_metrics]):.3f}")
print(f"Ortalama SENS: {np.mean([m['sensitivity'] for m in all_metrics]):.3f}")
print(f"Ortalama SPEC: {np.mean([m['specificity'] for m in all_metrics]):.3f}")
print(f"Ortalama F1: {np.mean([m['f1'] for m in all_metrics]):.3f}")


======================================================================FOLD 1/5======================================================================
  [Fold 1] [86 79] | LP: backbone dondurulmuş
  Pretrained: 126 katman yüklendi, 33 atlandı.
  Eğitilebilir: 61,922 / 62,249,218 parametre (0.1%)
    [Prob] min=0.344 max=0.749 | TP=17 FP=20 FN=3 TN=2
[lp | fold 1 | epoch 001] train=0.4354 val=0.2350 auc=0.4636 thr=0.419
  [Metrics] AUC:0.464 F1:0.596 SENS:0.850 SPEC:0.091 | COMPOSITE:0.5762
    [Prob] min=0.412 max=0.604 | TP=18 FP=17 FN=2 TN=5
[lp | fold 1 | epoch 002] train=0.2367 val=0.1731 auc=0.6023 thr=0.447
  [Metrics] AUC:0.602 F1:0.655 SENS:0.900 SPEC:0.227 | COMPOSITE:0.6633
    [Prob] min=0.372 max=0.625 | TP=18 FP=14 FN=2 TN=8
[lp | fold 1 | epoch 003] train=0.2039 val=0.1808 auc=0.6364 thr=0.420
  [Metrics] AUC:0.636 F1:0.692 SENS:0.900 SPEC:0.364 | COMPOSITE:0.7017
    [Prob] min=0.386 max=0.666 | TP=16 FP=16 FN=4 TN=6
[lp | fold 1 | epoch 004] train=0.2068 val=0.1834 auc=0.

KeyboardInterrupt: 

: 